<a href="https://colab.research.google.com/github/sinbbang/BigData/blob/main/10%EC%9E%A5_%EC%8B%A4%EC%8A%B501_%EC%98%A4%EB%8A%98%EC%9D%98%EB%82%A0%EC%94%A8%EB%A1%9C%EC%B6%9C%EA%B7%BC%EC%A4%80%EB%B9%84%ED%95%98%EA%B8%B0_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ☀️ 실습 ① 오늘의 날씨로 출근 준비하기

**10장 | 공공데이터와 API 연동 · Google Colab 실습**

**실습 목표:** 기상청 공공데이터를 받아 **출근 시간대의 기온·강수확률·강수형태**를 표로 정리하고, **우산 준비 안내**를 만들어 봅니다.

> ✅ **처음 실행할 때는 인증키가 필요 없는 `연습` 모드**를 사용합니다. 이후 공공데이터포털 인증키를 발급받아 `실제` 모드로 바꿀 수 있습니다.  
> ⚠️ 연습 모드 데이터는 **학습을 위해 만든 가상 예보**입니다. 실제 날씨가 아닙니다. 실제 모드의 데이터도 **예보**이므로 관측된 현재 날씨와 다를 수 있습니다.

**이 실습의 흐름:** 데이터 선택 → 인증키 확인 → API 요청 및 JSON 해부 → 필요한 값 추출 → 화면 표시·CSV 저장 → 데이터 검증

**공식 출처:** [공공데이터포털: 기상청_단기예보 조회서비스](https://www.data.go.kr/data/15084084/openapi.do)  
**API 주소:** `https://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getVilageFcst`


## STEP 1. 내가 확인할 지역과 출근 시간 정하기

먼저 아래 코드에서 `지역`, `출근시각`, `실습모드`를 바꿔 보세요. **코드를 전부 이해하지 않아도 됩니다.** 따옴표 안의 값만 수정하면 됩니다.

- `연습`: 인터넷/API 인증키 없이 가상 데이터로 바로 진행
- `실제`: 기상청 API로 실제 발표된 단기예보 요청 (인증키 필요)
- 위치는 입문용으로 제공된 **대표 예보 격자(nx, ny)** 입니다. 각 시·군 전체를 대표하는 것은 아닙니다.


In [1]:
# ===== 학생이 직접 바꾸는 설정값 =====
실습모드 = "연습"        # "연습" 또는 "실제" 중 하나를 선택합니다.
지역 = "서울"            # "서울", "대구", "부산" 중 하나를 선택합니다.
출근시각 = "09:00"       # 확인할 출근 시간입니다. 예: "08:00", "09:00"

# 기상청 단기예보 API는 위도·경도가 아니라 격자 좌표(nx, ny)를 사용합니다.
# 아래 좌표는 시청 부근 등 대표 위치를 위한 학습용 예시입니다.
지역좌표 = {
    "서울": {"nx": 60, "ny": 127},
    "대구": {"nx": 89, "ny": 90},
    "부산": {"nx": 98, "ny": 76},
}

# 올바른 지역을 선택했는지 먼저 확인합니다.
if 지역 not in 지역좌표:
    raise ValueError("지역은 서울, 대구, 부산 중 하나로 입력해 주세요.")

# 현재 설정을 눈으로 확인합니다.
print(f"선택 지역: {지역} / 출근시각: {출근시각} / 모드: {실습모드}")

선택 지역: 서울 / 출근시각: 09:00 / 모드: 연습


## STEP 2. 인증키 준비하기 (실제 모드에서만 필요)

1. [공공데이터포털](https://www.data.go.kr/data/15084084/openapi.do)에 로그인합니다.
2. **기상청_단기예보 조회서비스** 활용신청 후 서비스키를 확인합니다.
3. 아래 셀은 **실제 모드일 때만** 입력창을 보여 줍니다. 화면에 입력한 키를 코드에 저장하지 않습니다.

**보안:** 인증키가 들어 있는 화면을 캡처해 공유하거나 노트북 소스 코드에 키를 적어 두지 마세요. 실제 운영에는 Colab 비밀 관리 등 별도 보안 저장 방법을 사용합니다.


In [2]:
# 입력한 비밀 문자열이 화면에 그대로 보이지 않도록 getpass를 사용합니다.
from getpass import getpass

# 연습 모드에는 인증키가 필요 없습니다.
서비스키 = ""
if 실습모드 == "실제":
    # 공공데이터포털에서 받은 일반(Decoding) 인증키를 입력합니다.
    # 이미 인코딩된 키보다 디코딩 키를 사용하면 URL 이중 인코딩을 피하기 쉽습니다.
    서비스키 = getpass("공공데이터포털 일반(Decoding) 인증키를 입력하세요: ").strip()
    if not 서비스키:
        raise ValueError("실제 모드에서는 인증키를 입력해야 합니다.")
    print("인증키 입력을 확인했습니다. (보안을 위해 키 내용은 표시하지 않습니다.)")
else:
    print("연습 모드: 인증키 없이 진행합니다.")

연습 모드: 인증키 없이 진행합니다.


## STEP 3. JSON 응답 받아 보기

**API는 프로그램이 서버에 데이터를 요청하는 통로**입니다. JSON은 서버가 데이터를 보내는 형식 중 하나입니다.

실제 모드에서는 가장 최근에 **발표가 끝났을 것으로 예상되는** 단기예보 발표시각을 골라 요청합니다. 단기예보 발표시각은 일반적으로 `02·05·08·11·14·17·20·23시`이며, 게시 지연을 고려해 25분의 여유를 둡니다. 서버 상황에 따라 발표가 늦으면 오류 안내가 뜰 수 있습니다.


In [3]:
# 날짜와 시간은 한국 시간으로 처리합니다. Colab 서버가 외국에 있어도 한국 날짜를 사용합니다.
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import requests  # 인터넷 주소로 데이터를 요청하는 도구입니다.

# 실제 API가 제공하는 정해진 발표시각 목록입니다.
발표시각목록 = ["0200", "0500", "0800", "1100", "1400", "1700", "2000", "2300"]

def 최근_발표시각(현재시각):
    """한국 시간을 받아 안전하게 이용할 발표 날짜와 시간을 고릅니다."""
    # 발표 직후에는 데이터가 아직 서버에 없을 수 있으므로 25분 전을 기준으로 삼습니다.
    기준시각 = 현재시각 - timedelta(minutes=25)
    # 오늘 발표 중 현재 사용할 수 있는 시간만 찾습니다.
    사용가능 = [시각 for 시각 in 발표시각목록 if int(시각[:2]) <= 기준시각.hour]
    if 사용가능:
        return 기준시각.strftime("%Y%m%d"), 사용가능[-1]
    # 새벽 2시 이전이라면 전날 23시 발표 자료를 사용합니다.
    전날 = 기준시각 - timedelta(days=1)
    return 전날.strftime("%Y%m%d"), "2300"

현재 = datetime.now(ZoneInfo("Asia/Seoul"))
발표날짜, 발표시간 = 최근_발표시각(현재)
print(f"한국 현재 시각: {현재:%Y-%m-%d %H:%M}")
print(f"요청할 예보 발표 기준: {발표날짜} {발표시간}")

한국 현재 시각: 2026-11-13 18:38
요청할 예보 발표 기준: 20261113 1700


In [4]:
# 예제 JSON 데이터도 실제 기상청 응답과 같은 'response → body → items → item' 구조를 사용합니다.
# 이 값들은 날씨 교육용으로 만든 가상의 숫자이며 실제 관측·예보 자료가 아닙니다.
def 연습용_JSON_만들기(기준날짜):
    """오늘을 기준으로 출근 시간 예보를 담은 가상 JSON을 만듭니다."""
    # 시간대별로 날씨 숫자를 준비합니다. 각 시간은 기상청 형식(HHMM)으로 저장합니다.
    시간별값 = {
        "0800": {"TMP": "12", "POP": "20", "PTY": "0", "SKY": "3"},
        "0900": {"TMP": "13", "POP": "60", "PTY": "1", "SKY": "4"},
        "1200": {"TMP": "17", "POP": "30", "PTY": "0", "SKY": "3"},
        "1800": {"TMP": "14", "POP": "80", "PTY": "1", "SKY": "4"},
    }
    # 기상청 응답처럼 '한 행에 하나의 날씨 항목'을 넣습니다.
    가상항목 = []
    for 시각, 값모음 in 시간별값.items():
        for 항목코드, 값 in 값모음.items():
            가상항목.append({"fcstDate": 기준날짜, "fcstTime": 시각,
                         "category": 항목코드, "fcstValue": 값})
    # 아래 딕셔너리는 연습용 성공 응답 JSON입니다.
    return {"response": {"header": {"resultCode": "00", "resultMsg": "NORMAL_SERVICE"},
                         "body": {"totalCount": len(가상항목), "items": {"item": 가상항목}}}}

# 오늘 날짜를 한국 시간 기준으로 적습니다.
오늘날짜 = 현재.strftime("%Y%m%d")
API주소 = "https://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getVilageFcst"

if 실습모드 == "연습":
    # 인증키 없이도 공부할 수 있도록 가상 JSON을 생성합니다.
    응답JSON = 연습용_JSON_만들기(오늘날짜)
    데이터출처 = "학습용 가상 데이터 (실제 날씨 아님)"
else:
    # 요청 주소 뒤의 ?에 붙을 조건들(쿼리 매개변수)입니다.
    요청조건 = {
        "serviceKey": 서비스키, "pageNo": 1, "numOfRows": 1000,
        "dataType": "JSON", "base_date": 발표날짜, "base_time": 발표시간,
        "nx": 지역좌표[지역]["nx"], "ny": 지역좌표[지역]["ny"],
    }
    try:
        # timeout=20은 서버가 20초 안에 답하지 않으면 기다리기를 중단한다는 뜻입니다.
        서버응답 = requests.get(API주소, params=요청조건, timeout=20)
        서버응답.raise_for_status()  # HTTP 오류가 있으면 아래 오류 처리로 이동합니다.
        응답JSON = 서버응답.json()    # 응답이 JSON이면 Python 딕셔너리로 변환합니다.
    except (requests.RequestException, ValueError) as 오류:
        raise RuntimeError(f"API 요청에 실패했습니다. 인증키·인터넷·발표시각을 확인하세요. 상세: {오류}") from 오류
    데이터출처 = "기상청 단기예보 API (실제 발표 예보)"

# header가 성공(00)인지 반드시 확인합니다. HTTP 200만으로는 정상 자료라고 단정할 수 없습니다.
헤더 = 응답JSON.get("response", {}).get("header", {})
if str(헤더.get("resultCode", "")) != "00":
    raise RuntimeError(f"API 오류: {헤더.get('resultMsg', '알 수 없는 오류')} (코드 {헤더.get('resultCode')})")

# 자료 개수만 출력해 복잡한 JSON 전체가 화면을 가득 채우지 않게 합니다.
print("데이터 출처:", 데이터출처)
print("응답 코드:", 헤더.get("resultCode"), "/", 헤더.get("resultMsg"))
print("전체 응답 항목 수:", 응답JSON.get("response", {}).get("body", {}).get("totalCount"))

데이터 출처: 학습용 가상 데이터 (실제 날씨 아님)
응답 코드: 00 / NORMAL_SERVICE
전체 응답 항목 수: 16


## STEP 4. JSON에서 필요한 값 3개 찾기

날씨 응답에는 여러 종류의 숫자가 담겨 있습니다. 우리는 우선 **TMP(기온)**, **POP(강수확률)**, **PTY(강수형태)** 세 가지만 사용합니다.

- `TMP`: 예보 기온(℃)
- `POP`: 예보 강수확률(%)
- `PTY`: 강수형태 코드 (`0` 없음, `1` 비, `2` 비/눈, `3` 눈, `4` 소나기 등)

> **주의:** `POP`은 *비가 내릴 확률만* 뜻하는 게 아니라 **강수확률**입니다. 비뿐 아니라 눈도 포함될 수 있습니다.


In [5]:
# JSON은 여러 겹의 상자처럼 중첩되어 있습니다.
# 'response' 안의 'body', 그 안의 'items', 그 안의 'item' 목록을 꺼냅니다.
본문 = 응답JSON.get("response", {}).get("body", {})
항목들 = 본문.get("items", {}).get("item", []) if isinstance(본문, dict) else []
if isinstance(항목들, dict):
    항목들 = [항목들]  # 데이터가 한 건만 오는 응답도 안전하게 목록으로 처리합니다.
if not 항목들:
    raise RuntimeError("응답에 예보 자료가 없습니다. 다른 발표시각을 선택하거나 나중에 다시 시도하세요.")

# 실제 한 항목을 확인합니다. 이 행에는 날짜·시간·종류·값이 함께 있습니다.
print("JSON의 첫 번째 항목:")
print(항목들[0])

# 출근시각은 '09:00' 대신 API가 쓰는 '0900' 형식으로 바꿉니다.
출근시간코드 = 출근시각.replace(":", "")

# 선택 날짜(오늘)·시간대의 자료를 찾아 필요한 항목만 저장합니다.
선택값 = {}
for 항목 in 항목들:
    if 항목.get("fcstDate") == 오늘날짜 and 항목.get("fcstTime") == 출근시간코드:
        if 항목.get("category") in ["TMP", "POP", "PTY"]:
            선택값[항목["category"]] = 항목.get("fcstValue")

print(f"{지역} {출근시각} 예보에서 추출한 값:", 선택값)
if not 선택값:
    print("안내: 해당 날짜·시간의 예보가 없습니다. 아래 단계에서 다른 시간대도 살펴보세요.")

JSON의 첫 번째 항목:
{'fcstDate': '20261113', 'fcstTime': '0800', 'category': 'TMP', 'fcstValue': '12'}
서울 09:00 예보에서 추출한 값: {'TMP': '13', 'POP': '60', 'PTY': '1'}


## STEP 5. 출근 준비 안내표 만들기

**중요:** '우산 권장'은 기상청 공식 행동지침이 아니라 **수업에서 정의한 간단한 규칙**입니다. 강수형태가 있거나 강수확률이 50% 이상이면 우산을 권장해 봅니다. 실제 판단에는 일기예보·이동 경로 등을 함께 확인하세요.


In [6]:
# 여러 행을 표로 보여 주기 위해 pandas를 사용합니다. Colab에 기본 설치되어 있습니다.
import pandas as pd
from IPython.display import display

# 예보 항목이 시각별로 나뉘어 있으므로 시간을 기준으로 다시 묶습니다.
시간별예보 = {}
for 항목 in 항목들:
    if 항목.get("fcstDate") != 오늘날짜:
        continue  # 오늘 예보가 아니면 건너뜁니다.
    시각 = 항목.get("fcstTime", "")
    if 시각 not in 시간별예보:
        시간별예보[시각] = {}
    시간별예보[시각][항목.get("category")] = 항목.get("fcstValue")

# 강수형태 코드에 사람이 읽기 쉬운 이름을 붙입니다.
강수형태이름 = {"0": "없음", "1": "비", "2": "비/눈", "3": "눈", "4": "소나기"}

def 우산안내(강수확률, 강수형태):
    """수업용 기준에 따라 우산 준비 여부를 안내합니다."""
    # 강수확률은 숫자가 없을 수 있으므로 안전하게 변환합니다.
    try:
        확률 = float(강수확률)
    except (TypeError, ValueError):
        확률 = None
    # 강수형태가 없거나 알 수 없는 경우 임의로 '맑음'으로 단정하지 않습니다.
    if 확률 is None and 강수형태 is None:
        return "정보 부족"
    if (강수형태 is not None and 강수형태 != "0") or (확률 is not None and 확률 >= 50):
        return "우산 준비 권장"
    if 확률 is not None:
        return "강수 가능성 확인"
    return "정보 부족"

# 행(한 시간의 날씨)을 차례대로 만듭니다.
표행 = []
for 시각, 값 in sorted(시간별예보.items()):
    # 숫자/코드가 비어 있으면 억지로 0으로 바꾸지 않습니다.
    기온 = 값.get("TMP")
    확률 = 값.get("POP")
    강수형태 = 값.get("PTY")
    표행.append({
        "지역": 지역,
        "예보날짜": f"{오늘날짜[:4]}-{오늘날짜[4:6]}-{오늘날짜[6:]}",
        "예보시각": f"{시각[:2]}:{시각[2:]}",
        "기온(℃)": 기온 if 기온 is not None else "자료 없음",
        "강수확률(%)": 확률 if 확률 is not None else "자료 없음",
        "강수형태": 강수형태이름.get(str(강수형태), "자료 없음/기타"),
        "출근 준비 안내": 우산안내(확률, 강수형태),
    })

# DataFrame은 데이터를 엑셀과 비슷한 표 형태로 다루는 기능입니다.
날씨표 = pd.DataFrame(표행)
if 날씨표.empty:
    print("오늘 예보가 없습니다. 실제 모드라면 API 발표시각과 예보 범위를 확인하세요.")
else:
    print(f"[{데이터출처}] {지역}의 오늘 시간별 날씨")
    display(날씨표.head(24))  # 화면에는 최대 24행만 보여 줍니다.

[학습용 가상 데이터 (실제 날씨 아님)] 서울의 오늘 시간별 날씨


,지역,예보날짜,예보시각,기온(℃),강수확률(%),강수형태,출근 준비 안내
0,서울,2026-11-13,08:00,12,20,없음,강수 가능성 확인
1,서울,2026-11-13,09:00,13,60,비,우산 준비 권장
2,서울,2026-11-13,12:00,17,30,없음,강수 가능성 확인
3,서울,2026-11-13,18:00,14,80,비,우산 준비 권장


In [7]:
# 출근시간 1개 행을 골라 읽기 쉬운 문장으로 보여 줍니다.
출근행 = 날씨표[날씨표["예보시각"] == 출근시각] if not 날씨표.empty else pd.DataFrame()
if 출근행.empty:
    print(f"{오늘날짜} {출근시각} 예보가 없습니다. 08:00·09:00 또는 다른 시간을 선택해 보세요.")
else:
    # iloc[0]은 표에서 첫 번째 행을 꺼내는 표현입니다.
    한행 = 출근행.iloc[0]
    print(f"\n📍 {지역} {출근시각} 출근 준비 안내")
    print(f"  기온: {한행['기온(℃)']}℃")
    print(f"  강수확률: {한행['강수확률(%)']}%")
    print(f"  강수형태: {한행['강수형태']}")
    print(f"  안내: {한행['출근 준비 안내']}")
    print("  ※ 간단한 학습용 판단 규칙입니다. 공식 행동지침이 아닙니다.")


📍 서울 09:00 출근 준비 안내
  기온: 13℃
  강수확률: 60%
  강수형태: 비
  안내: 우산 준비 권장
  ※ 간단한 학습용 판단 규칙입니다. 공식 행동지침이 아닙니다.


## STEP 6. 결과 검증하고 CSV 파일로 저장하기

**검증 세 가지**

1. 데이터 출처가 `연습`이면 가상자료인지 확인했나요?
2. 선택한 지역·날짜·시간의 행이 맞나요?
3. 응답에 값이 없는데 `0`으로 잘못 표시한 곳은 없나요?

> CSV는 엑셀에서도 열 수 있는 표 형태의 파일입니다. 한글이 깨지지 않도록 `utf-8-sig` 인코딩을 사용합니다.


In [8]:
# 데이터의 행 수를 확인하고 파일로 저장합니다.
if 날씨표.empty:
    print("저장할 날씨 데이터가 없습니다. API 응답 또는 출근시간을 확인하세요.")
else:
    # 파일명에도 날짜와 지역을 넣어 나중에 찾기 쉽게 합니다.
    파일명 = f"{오늘날짜}_{지역}_출근준비_날씨표.csv"
    날씨표.to_csv(파일명, index=False, encoding="utf-8-sig")
    print(f"저장 완료: {파일명} (총 {len(날씨표)}개 시간대)")
    print("Colab 왼쪽 '파일' 탭에서 파일을 확인하거나 다음 셀로 내려받으세요.")

저장 완료: 20261113_서울_출근준비_날씨표.csv (총 4개 시간대)
Colab 왼쪽 '파일' 탭에서 파일을 확인하거나 다음 셀로 내려받으세요.


In [9]:
# 이 셀은 Colab에서 실행하면 CSV 파일을 내 컴퓨터로 다운로드합니다.
# 로컬 Jupyter 등 Colab이 아닌 환경에서는 파일이 이미 저장되었다는 안내만 출력합니다.
if not 날씨표.empty:
    try:
        from google.colab import files  # Colab 전용 다운로드 도구입니다.
        files.download(파일명)         # 브라우저에 다운로드를 요청합니다.
    except ImportError:
        print(f"Colab 외부 환경입니다. 현재 작업 폴더의 '{파일명}' 파일을 확인하세요.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## ✍️ 직접 바꿔 보는 연습문제

1. **기본:** STEP 1의 `지역 = "서울"`을 `"대구"`로 바꾼 뒤 전체 셀을 다시 실행해 보세요. **연습 모드는 지역을 바꿔도 같은 가상 날씨**가 표시됩니다. 실제 지역별 차이를 보려면 실제 모드가 필요합니다.
2. **응용:** STEP 1의 `출근시각`을 `"08:00"`으로 바꾸고, 09:00 결과와 비교해 보세요.
3. **도전:** STEP 5의 `우산안내` 함수에서 `확률 >= 50`을 `확률 >= 30`으로 바꿔 보세요. 안내 결과가 어떻게 달라지는지 설명해 보세요.

### 생각해 보기

- 같은 데이터라도 우산 권장 기준이 다르면 안내가 왜 달라질까요?
- API가 일시적으로 연결되지 않으면 사용자에게 어떤 메시지를 보여 주는 게 좋을까요?
- **예보**와 **관측값**은 어떤 차이가 있을까요?

### 참고: 이 노트북의 범위

- 실제 모드는 **기상청 단기예보**를 요청하며, 실시간 관측값을 조회하지는 않습니다.
- 샘플 데이터는 학습 목적의 가상 수치입니다.
- 지역 좌표는 대표 격자이며 실제 주소·동 단위 위치를 자동 변환하지 않습니다.
- 인증키는 노트북에 저장하지 마세요. 키 입력 이후 `변수`에 잠시 남으므로 공유 전에 런타임을 초기화하세요.

**강의자료 연결:** 10장 17~24쪽의 데이터 찾기 → 인증키 → JSON 응답 → 세 값 추출 → 화면 표시·검증 흐름을 Colab으로 구현했습니다.
